# oneway・mutual 본실험 — A / B / C1

이번 고정 실험은 **oneway와 mutual 모두**, 조건은 **A·B·C1**입니다. C0·C2는 실행하지 않습니다.
네 사람이 5씬씩 맡아 총 20씬 × 3조건 = 60회입니다. 한 사람은 자기 씬의 모든 조건을 실행합니다.
VLM R5 + motion7k를 고정하고 A/B/C 모두 같은 VLA를 사용합니다. A는 전체 지시를 받으며 VLM을 호출하지 않습니다.
부분 작업 학습 VLA이므로 A의 전체 지시는 학습 분포와 다릅니다. 전체 정리 성공률 60%가 검증된 모델은 아닙니다.

Colab GPU를 연결하고 **위에서부터 실행**하세요. 설정에서 `PART`를 고르고, 실행 목록을 확인한 뒤 `RUN_EXPERIMENT=True`로 바꿉니다.
씬·전략 선택이나 디버깅에 최종 test 씬의 점수를 쓰지 않습니다. C1은 시간과 실패 대응을 고려해 사전에 선택했으며 성능 1위가 입증된 것은 아닙니다.
자세한 안내: 저장소의 `research/3_colab_guide.md`.

In [ ]:
import os, sys, json, platform, subprocess, shutil, uuid, random
from pathlib import Path
IS_COLAB = platform.system() == 'Linux' and Path('/content').is_dir()
PART = 'part2_oneway_b'   # part1_oneway_a / part2_oneway_b / part3_mutual_a / part4_mutual_b
TEAM = PART
REPO_URL = 'https://github.com/semin126/VLA2026.git'
REPO_REF = 'instructor/runtime-conda-fixes'
SOURCE_ROOT = os.environ.get('FPV2_SOURCE', '')  # 보통 비워 둡니다. 명시한 로컬/업로드 코드만 사용
INSTALL = IS_COLAB and os.environ.get('FPV2_SKIP_INSTALL') != '1'
USE_DRIVE = IS_COLAB and os.environ.get('FPV2_NO_DRIVE') != '1'
PYTHON_OVERRIDE = os.environ.get('FPV2_PYTHON', '')
WORK = Path(os.environ.get('FPV2_WORK', '/content/Final_project_v2' if IS_COLAB else 'outputs/final_project_v2'))
MODEL_SOURCE = 'local'  # 모델 누락 시 Hub의 다른 모델로 자동 대체하지 않음
MODELS_DIR = os.environ.get('FPV2_MODELS_DIR', 'models')
DOWNLOAD_MODELS = True
RUN_EXPERIMENT = False  # 준비·실행 목록 확인 후 True로 변경
SECONDS, MAX_STEPS, ACTION_STEPS, OBSERVER_STEPS, MAX_CALLS, SEED = 300, 6000, 240, 60, 12, 42
PARTS = ('part1_oneway_a', 'part2_oneway_b', 'part3_mutual_a', 'part4_mutual_b')
assert PART in PARTS, 'PART를 네 묶음 중 하나로 골라 주세요.'

## 코드・Python 3.12・저장 위치

Colab은 지정 브랜치를 새 작업 폴더로 받습니다. 기존 체크아웃을 자동 변경하지 않습니다.
VS Code 로컬에서는 `VLA2026` conda 커널을 선택하세요. 모델 추론은 아래에 표시되는 Execution Python에서 실행합니다.
Drive 연결을 허용하면 **매 trial 명령 종료 후** 결과를 해시 확인하여 백업합니다. 실행 도중 연결이 끊긴 trial은 미백업일 수 있습니다.

In [ ]:
GIT = shutil.which('git') or str(Path(os.environ.get('ProgramFiles', 'C:/Program Files'))/'Git/cmd/git.exe')
if SOURCE_ROOT:
    PROJECT = Path(SOURCE_ROOT).expanduser().resolve()
elif IS_COLAB:
    checkout = Path('/content')/('fpv2_c1_'+uuid.uuid4().hex[:8])
    subprocess.run([GIT, 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, str(checkout)], check=True)
    PROJECT = checkout/'vla_pipeline'
else:
    candidates = [p for base in (Path.cwd(), *Path.cwd().parents)
                  for p in (base, base/'vla_pipeline', base/'2026/VLA2026/vla_pipeline')]
    PROJECT = next((p for p in candidates if (p/'scripts/final_project_v2.py').is_file()), None)
    if PROJECT is None:
        raise RuntimeError('vla_pipeline 폴더에서 열거나 FPV2_SOURCE에 코드 위치를 지정하세요.')
PROJECT = PROJECT.resolve()
sys.path.insert(0, str(PROJECT))
from vlapipe.final_project_v2.notebook_runtime import NotebookRun, prepare_python, source_version
if not WORK.is_absolute(): WORK = PROJECT/WORK
if USE_DRIVE:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/VLA2026/Final_project_v2')
else:
    BACKUP = None
run = NotebookRun(PROJECT, WORK, BACKUP, TEAM)
PY = prepare_python(PROJECT, WORK, install=INSTALL, python_override=PYTHON_OVERRIDE)
run.python = PY
VERSION = source_version(PROJECT)
(run.root/'source_version.json').write_text(json.dumps(VERSION, indent=2), encoding='utf-8')
print('SOURCE VERSION:', VERSION)
print('PROJECT:', PROJECT, '\nExecution Python:', PY, '\nResults:', run.root, '\nBackup:', run.backup)
run.run('scripts/check_env.py')
subprocess.run([PY, '-c', "import torch,cv2; assert torch.cuda.is_available(), 'GPU 런타임을 연결하세요'; print('GPU:', torch.cuda.get_device_name(0), 'OpenCV:', cv2.__version__)"], cwd=PROJECT, check=True)

## 배포 모델 받기

고정 명세의 **VLM R5, VLA motion7k 두 역할**만 받습니다(압축 파일 약 1.9GB).
SHA256을 검사하고 같은 버전은 유지합니다. 이전 모델은 새 파일 검증 후 `models/_previous/`에 보관합니다.
이번 실험에는 `vla_single`을 받지 않으며 Hub 모델로 자동 대체하지 않습니다.

In [ ]:
if DOWNLOAD_MODELS:
    run.run('scripts/download_models.py', '--manifest', 'models/release_manifest_deadline_20261106.json',
            '--roles', 'vlm,vla_hier', '--models-dir', MODELS_DIR, '--upgrade')
else:
    print('다운로드를 건너뜁니다. 다음 실행 전 검사가 두 역할의 고정 버전을 확인합니다.')

## 실행 전 검사・담당 씬 5개 확인

이 셀은 최종 씬에서 모델을 실행하지 않습니다. 파일·고정 상태 해시·전략·모델 버전·실행 목록을 확인합니다.
개인당 담당 계열 하나, 네 사람의 결과를 합치면 oneway 10씬 + mutual 10씬입니다.

In [ ]:
import hashlib
from vlapipe.final_project_v2.common import digest
SUITE = PROJECT/'outputs/shared_scenes/instructor_test_v2/colab_split4'/PART
C_STRATEGY = PROJECT/'final_project_v2/student_strategy_c1_v2.py'
assert (SUITE/'suite.json').is_file(), '씬 묶음이 없습니다. 최신 강사 브랜치를 확인하세요.'
assert C_STRATEGY.is_file(), 'C1 v2 전략 파일이 없습니다.'
strategy_text = C_STRATEGY.read_text(encoding='utf-8')
assert 'MODE = "C1"' in strategy_text or "MODE = 'C1'" in strategy_text, '선택한 파일은 C1이어야 합니다.'
registry = json.loads((SUITE/'suite.json').read_text(encoding='utf-8'))
selected = [r for r in registry['scenes'] if r['split']=='test']
assert len(selected)==5, '담당 묶음은 test 씬 5개여야 합니다.'
EXPECTED_FAMILY = 'oneway' if '_oneway_' in PART else 'mutual'
PLAN = []
for row in selected:
    scene = (SUITE/row['path']).resolve()
    spec = json.loads(scene.read_text(encoding='utf-8'))
    assert EXPECTED_FAMILY in row['family'], '담당 계열과 씬이 다릅니다.'
    assert digest(spec['initial_state'])==spec['state_sha256']==row['state_sha256'], '씬 고정 상태가 바뀌었습니다.'
    order = list('ABC')
    random.Random(str(SEED)+row['path']).shuffle(order)
    for condition in order:
        PLAN.append({'scene':str(scene), 'scene_id':scene.stem, 'family':EXPECTED_FAMILY,
                     'condition':condition, 'strategy':'C1' if condition=='C' else None, 'seed':SEED,
                     'output':f'{scene.stem}/{SEED}/{condition}'})
assert len({(r['scene'],r['condition']) for r in PLAN})==15
sys.path.insert(0, str(PROJECT/'scripts'))
from download_models import verify_bundle
model_root = Path(MODELS_DIR)
if not model_root.is_absolute(): model_root = PROJECT/model_root
manifest = json.loads((PROJECT/'models/release_manifest_deadline_20261106.json').read_text(encoding='utf-8'))
for role in ('vlm','vla_hier'):
    entry = manifest['models'][role]
    verify_bundle(model_root/role/'pretrained_model', role, entry.get('bundle_release',manifest['release']))
OUT = run.root/('test_shared_c1_'+PART)
print('계열:', EXPECTED_FAMILY, '| 씬:', len(selected), '| 실행:', len(PLAN), '| C 전략: C1 v2')
for row in PLAN: print(row['scene_id'], row['condition'], row['strategy'] or '', 'seed', row['seed'])
print('C1 SHA256:', hashlib.sha256(C_STRATEGY.read_bytes()).hexdigest())
print('출력:', OUT)

## 본실험 실행 — A・B・C1

설정 셀에서 `RUN_EXPERIMENT=True`로 바꾼 뒤 이 셀을 실행합니다. 설정 셀만 다시 실행할 때 `PART`를 바꾸지 마세요.
15회 × 최대 300초 + 모델 로딩·녹화·백업 시간입니다. 시간은 GPU와 Drive 속도에 따라 달라집니다.
조건 순서는 씬마다 고정된 seed로 섞으며, 모든 조건은 같은 frozen state에서 시작합니다.
한 Colab에서 병렬 실행하지 않습니다. 이미 끝난 결과는 낮은 점수라는 이유로 다시 돌리지 않습니다.
이 셀 재실행 시 기존 결과를 덮어쓰지 않고 중단합니다. 기술적 오류 재실행은 조교와 대상 trial을 확인하세요.

In [ ]:
if RUN_EXPERIMENT:
    OUT.mkdir(parents=True, exist_ok=False)
    (OUT/'schedule.json').write_text(json.dumps({'trials':PLAN, 'source':VERSION,
        'model_manifest':manifest, 'c_strategy_sha256':hashlib.sha256(C_STRATEGY.read_bytes()).hexdigest(),
        'budgets':{'seconds':SECONDS, 'max_steps':MAX_STEPS, 'action_steps':ACTION_STEPS,
                   'observer_steps':OBSERVER_STEPS, 'max_calls':MAX_CALLS}}, indent=2), encoding='utf-8')
    failures = []
    for index, row in enumerate(PLAN, 1):
        print(f"[{index}/{len(PLAN)}]", row['scene_id'], row['condition'], flush=True)
        target = OUT/row['output']
        args = ['trial', '--scene', row['scene'], '--condition', row['condition'], '--out', target,
                '--seconds', SECONDS, '--max-steps', MAX_STEPS, '--action-steps', ACTION_STEPS,
                '--observer-steps', OBSERVER_STEPS, '--max-calls', MAX_CALLS, '--seed', row['seed'],
                '--model-source', MODEL_SOURCE, '--models-dir', MODELS_DIR, '--shared-vla', '--final-test']
        if row['condition']=='C': args += ['--strategy', C_STRATEGY]
        try:
            run.run('scripts/final_project_v2.py', *args)
        except subprocess.CalledProcessError as exc:
            failures.append({**row, 'returncode':exc.returncode})
            print('실행 오류: 로그를 보관하고 다음 trial을 진행합니다.')
        finally:
            (OUT/'launch_errors.json').write_text(json.dumps(failures, indent=2), encoding='utf-8')
            run.sync()
    print('명령 종료. 아래 결과 확인 셀에서 15개 보고서와 오류 유무를 확인하세요.', OUT)
else:
    print('아직 본실험을 실행하지 않았습니다. 목록 확인 후 RUN_EXPERIMENT=True로 바꾸세요.')

## 결과 확인・집계・영상・전달

`completed`는 코드가 정상 종료했다는 뜻이며 정리 성공을 뜻하지 않습니다. 성공 여부는 `scene_success`와 ACC/F1을 보세요.
15개가 없거나 `error`가 있다면 오류 로그를 함께 전달하세요. oneway와 mutual의 평균은 구분해 계산합니다.
각 trial 폴더의 `process.mp4`, `plan_*.json`, `events.jsonl`, `report.json`으로 동작과 재계획을 확인할 수 있습니다.
아래 압축 파일과 Drive의 `test_shared_c1_<PART>` 폴더를 조교에게 전달합니다. 기존 D 결과는 별도로 보관합니다.

In [ ]:
if OUT.is_dir():
    rows = []
    for file in sorted(OUT.rglob('report.json')):
        report = json.loads(file.read_text(encoding='utf-8'))
        rows.append((file.parent.parent.parent.name, report['condition'], report['status'],
                     report['final']['acc'], report['final']['macro_f1'], report['final']['scene_success'],
                     round(report['summary']['total_wall_seconds'],2), report['summary']['planner_calls']))
    print('씬 / 조건 / 상태 / ACC / F1 / 씬 완료 / 시간(초) / VLM 호출')
    for row in rows: print(*row)
    print(len(rows), '/ 15개 보고서')
    errors = json.loads((OUT/'launch_errors.json').read_text()) if (OUT/'launch_errors.json').is_file() else []
    if len(rows)!=15 or errors or any(r[2]!='completed' for r in rows):
        print('누락 또는 오류가 있습니다. 조교에게 로그와 함께 알려 주세요.', errors)
    if rows:
        run.run('scripts/final_project_v2.py', 'aggregate', '--root', OUT)
    run.sync()
    archive = shutil.make_archive(str(run.root/OUT.name), 'zip', OUT)
    print('전달할 압축 파일:', archive)
    print('Drive 결과 폴더:', run.backup/OUT.name if run.backup else 'Drive 미연결: 로컬 파일을 내려받으세요.')
else:
    print('본실험 결과가 아직 없습니다.')